In [20]:
import pandas as pd
import numpy as np
import os
from sklearn.neighbors import KNeighborsClassifier
from PIL import Image
import matplotlib.pyplot as plt
from google.colab import files

import os
import glob
import numpy as np
import cv2
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split   # dataset splitting
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
# configurations
IMG_SIZE      = 96      # Every image is resized to 96×96 pixels before training
CHANNELS      = 1       # 1 = grayscale
BATCH_SIZE    = 32
EPOCHS        = 30      # Maximum number of complete passes over the training data.
LEARNING_RATE = 1e-3    # How large a step the optimiser takes each iteration.
TEST_SPLIT    = 0.15    # 15 % of data held out as the final test set
RANDOM_SEED   = 123      # Fixes all random operations so results are reproducible

In [29]:
mainPath = "/content/drive/MyDrive/ACMResearchDataset/indoor_dataset/"

traversablePath = mainPath + "traversibleTrue"
untraversablePath = mainPath + "traversibleFalse"


traversableFiles = sorted(os.listdir(traversablePath))
untraversableFiles = sorted(os.listdir(untraversablePath))

traversableFilePaths = [traversablePath + "/" + file for file in traversableFiles]
untraversableFilePaths = [untraversablePath + "/" + file for file in untraversableFiles]

print(len(traversableFilePaths))
print(len(untraversableFilePaths))
# img = Image.open(firstFilePath)

numTrainingImages = 3000

traversableFilePathsTraining = traversableFilePaths[:numTrainingImages]
untraversableFilePathsTraining = untraversableFilePaths[:numTrainingImages]

traversableFilePathsTesting = traversableFilePaths[numTrainingImages:]
untraversableFilePathsTesting = untraversableFilePaths[numTrainingImages:]

print(len(traversableFilePathsTraining))
print(len(untraversableFilePathsTraining))

traversableFilePathsTraining = traversableFilePaths[:numTrainingImages:4]
untraversableFilePathsTraining = untraversableFilePaths[:numTrainingImages:4]

traversableFilePathsTesting = traversableFilePaths[numTrainingImages::4]
untraversableFilePathsTesting = untraversableFilePaths[numTrainingImages::4]

print(len(traversableFilePathsTraining))
print(len(untraversableFilePathsTraining))


filePathsTraining = traversableFilePathsTraining + untraversableFilePathsTraining
labelsTraining = [1] * len(traversableFilePathsTraining) + [0] * len(untraversableFilePathsTraining)

filePathsTesting = traversableFilePathsTesting + untraversableFilePathsTesting
labelsTesting = [1] * len(traversableFilePathsTesting) + [0] * len(untraversableFilePathsTesting)

3906
3129
3000
3000
750
750


In [ ]:
train_ds_x = np.zeros((len(filePathsTraining), IMG_SIZE, IMG_SIZE), dtype = np.float32)
train_ds_y = np.array(labelsTraining)

test_ds_x = np.zeros((len(filePathsTesting), IMG_SIZE, IMG_SIZE), dtype=np.float32)
test_ds_y = np.array(labelsTesting)

def addedImages(imageSet, filePaths):
  for i, path in enumerate(filePaths):
    img = Image.open(path)
    preprocessedImage = img.convert("L").resize((IMG_SIZE, IMG_SIZE))
    imageSet[i] = np.array(preprocessedImage)

  return imageSet

train_ds_x = addedImages(train_ds_x, filePathsTraining)
test_ds_x  = addedImages(test_ds_x, filePathsTesting)

np.savez("trainDatasetIndoors.npz", x=train_ds_x, y=train_ds_y)
np.savez("testDatasetIndoors.npz", x=test_ds_x, y=test_ds_y)

files.download("trainDatasetIndoors.npz")
files.download("testDatasetIndoors.npz")